# BioBERT Training

Notebook for interactively training and evaluating the BioBERT transformer.

In [ ]:
import numpy as np
import torch
from transformers import TrainingArguments, Trainer
import pandas as pd
import os
from transformers import DataCollatorWithPadding, AdamW, EarlyStoppingCallback, Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification
from transformers.optimization import get_polynomial_decay_schedule_with_warmup
import torch.nn.functional as F
from sklearn.isotonic import IsotonicRegression
import random
import biobert_train_src
from datasets import Dataset
import shutil

tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")
model = AutoModelForSequenceClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.1", num_labels=2,ignore_mismatched_sizes=True)


def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multi-GPU
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Set a global seed
set_seed(42)

split = {
    "train": [0,1,2],
    "dev": [3],
    "test": [4]
}

dataset = "Analgesics-induced_acute_liver_failure"

df_train, df_dev, df_test = biobert_train_src.get_split(split, dataset)

train_data = Dataset.from_pandas(df_train)
test_data = Dataset.from_pandas(df_test)
dev_data = Dataset.from_pandas(df_dev)

def preprocess_function(examples):
    # Lowercase the text separately
    examples["Temp_sentence"] = [sentence.lower() for sentence in examples["Temp_sentence"]]
    return tokenizer(examples["Temp_sentence"], truncation=True, max_length=128)

tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)
tokenized_dev = dev_data.map(preprocess_function, batched=True)

trainer = biobert_train_src.train_transformer(tokenizer, model, tokenized_train, tokenized_dev)

prob_dev, prob_test = biobert_train_src.predict(trainer, tokenized_dev, tokenized_test)

prob_test_cal = biobert_train_src.calibration(prob_dev, df_dev["label"].tolist(), prob_test)

 # delete results and logs dir
shutil.rmtree("results")
shutil.rmtree("logs")
shutil.rmtree("mlruns")
shutil.rmtree("wandb")

df_res = pd.DataFrame()
df_res.index = df_test.index
df_res["biobert_temp"] = prob_test
df_res["biobert_temp_cal"] = prob_test_cal
df_res["label"] = df_test["label"]

df_res.to_csv(os.path.join("/home/hsdslab/Documents/Csabi/Pharma_crossval/DeepCausalPV-master-main/dat", dataset,
                           "proc", "cross_val_biobert_temp", f"df_res{split["dev"][0]}{split["test"][0]}.csv"))

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/21907 [00:00<?, ? examples/s]

Map:   0%|          | 0/7461 [00:00<?, ? examples/s]

Map:   0%|          | 0/7291 [00:00<?, ? examples/s]

/home/hsdslab/anaconda3/lib/python3.12/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
/home/hsdslab/anaconda3/lib/python3.12/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/home/hsdslab/Documents/Csabi/Pharma_crossval/DeepCausalPV-master-main/src/Analgesics-induced_acute_liver_failure/biobert_train_src.py:72: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  optimizers=(optimizer, None),  # Pass custom optimizer (scheduler set manually below)
max_steps is given, it will override any value given in num_train_epochs
wand

Step,Training Loss,Validation Loss
200,0.560100,0.519457
400,0.506300,0.481161


In [3]:
import numpy as np

possible_splits = []

for i in range(5):
    for j in range(i+1,5):
        dicti = {
            "test": [i],
            "dev": [j],
            "train": [x for x in range(5) if x not in [i,j]]
        }
        possible_splits.append(dicti)

In [8]:
possible_splits[4]

{'test': [1], 'dev': [2], 'train': [0, 3, 4]}